# Evaluate the traffic classifier
Evaluate the independent test split once model settings are selected using validation data. No training occurs here.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'traffic_training').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the Train folder or its notebooks subfolder.')
sys.path.insert(0, str(ROOT))
from IPython.display import display, Image


In [ ]:
MODEL = ROOT / 'outputs' / 'cv' / 'classifier.pt'
TEST_DATA = ROOT / 'datasets' / 'traffic' / 'test'
OUTPUT = ROOT / 'outputs' / 'cv-test'
IMAGE_SIZE = 224  # Must match the training metadata and demo ROI resolution.
BATCH = 32
DEVICE = 'cpu'


In [ ]:
from traffic_training.classifier import evaluate_classifier
metrics = evaluate_classifier(MODEL, TEST_DATA, OUTPUT, image_size=IMAGE_SIZE, batch=BATCH, device=DEVICE)
display({key: value for key, value in metrics.items() if key != 'classification_report'})


In [ ]:
import pandas as pd
display(pd.DataFrame(metrics['classification_report']).T)
display(Image(filename=str(OUTPUT / 'confusion_matrix.png')))
display(pd.read_csv(OUTPUT / 'predictions.csv').head(20))


Top-k uses min(5, number of classes), so it is trivial for small class counts. Inspect top-1, macro F1, per-class recall and the confusion matrix. The reported latency measures a warmed-up batch forward pass per image; it excludes video decoding, cropping, resize and network delay. This does not establish real-time demo FPS. The standalone equivalent is `scripts/evaluate_classifier.py`.